### Entrainer un nouveau model sentencepiece et expandre le tokenizer de nllb

In [3]:
import sentencepiece as spm

# Chemin vers votre fichier de texte brut (ex: corpus.txt)
input_corpus = "../Data/full_dataset/arabe_tchad3.txt"
model_prefix = "shu_Latn_sp"

# Entraînement du tokenizer
spm.SentencePieceTrainer.train(
    input=input_corpus,
    model_prefix=model_prefix,
    vocab_size=8000,          # Taille cible du vocabulaire pour votre langue
    model_type="BPE",         # NLLB utilise le format BPE (Byte Pair Encoding)
    character_coverage=0.9995,# Couverture des caractères (0.9995 recommandé pour les langues à alphabet)
    byte_fallback=True        # Crucial pour éviter les tokens inconnus (<unk>)
)


I0000 00:00:1791357544.318319   47230 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: ../Data/full_dataset/arabe_tchad3.txt
  input_format: 
  model_prefix: shu_Latn_sp
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_

True

### Extraire et nettoyer les news tokens

In [4]:
# Charger les tokens générés par SentencePiece
new_tokens = []
with open(f"{model_prefix}.vocab", "r", encoding="utf-8") as f:
    for line in f:
        token = line.split("\t")[0]
        # On ignore les tokens de contrôle par défaut de SentencePiece
        if token not in ["<unk>", "<s>", "</s>", "<pad>"]:
            new_tokens.append(token)

print(f"Nombre de tokens extraits : {len(new_tokens)}")


Nombre de tokens extraits : 7997


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_id = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Récupérer le vocabulaire actuel de NLLB
nllb_vocab = tokenizer.get_vocab()

# Filtrer les nouveaux tokens pour ne garder que ceux absents de NLLB
tokens_to_add = [t for t in new_tokens if t not in nllb_vocab]

# 1. Ajouter le code de la nouvelle langue (ex: 'new_Latn')
new_lang_code = "shu_Latn"
tokenizer.add_special_tokens({"additional_special_tokens": [new_lang_code]})

# 4. Mettre à jour les dictionnaires internes de NLLB pour associer le token à une langue
# Cela évite les erreurs lors de la configuration de `src_lang` ou `tgt_lang`
tokenizer.added_tokens_encoder[new_lang_code] = len(tokenizer) - 1
tokenizer.added_tokens_decoder[len(tokenizer) - 1] = new_lang_code

# 2. Ajouter les nouveaux tokens textuels issus de SentencePiece
tokenizer.add_tokens(tokens_to_add)

# 3. Redimensionner les embeddings du modèle
model.resize_token_embeddings(len(tokenizer))

# 4. Sauvegarder le tout
tokenizer.save_pretrained("../expanded_model_and_tokenizer/nllb-custom-tokenizer")
model.save_pretrained("../expanded_model_and_tokenizer/nllb-custom-tokenizer")

print(f"Terminé ! {len(tokens_to_add)} nouveaux tokens et la langue '{new_lang_code}' ont été ajoutés.")


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 3267.28it/s]
